In [1]:
%pip install -q syncode

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re

from syncode import Syncode

In [3]:
grammar = r""" 
start: " " number " " op " " number 

number: /-?(0|[1-9][0-9]*)\.[0-9]+/

op: "+" | "-" | "*" | "/"
""".strip()

model_name = "microsoft/phi-2"

# Load the Syncode augmented model
llm = Syncode(
    model=model_name,
    grammar=grammar,
    parse_output_only=True,
    max_new_tokens=50,
    device="cpu",
    device_map="cpu",
)

[2026-10-26 19:02:02,745-root] - Loading model microsoft/phi-2 with device:cpu, device_map:cpu, torch_dtype:torch.bfloat16


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
PROMPT = """
Q: What is 327. multiplied by 11.0?
A: 327.0 * 11.0

Q: What is 45.1 plus 23.54?
A: 45.1 + 23.54

Q: What is 120.4 divided by 4.0?
A: 120.4 / 4.0

Q: What is .1 minus -6.7?
A: 0.1 - -6.7

Q: What is -15 plus 3.2?
A: -15.0 + 3.2

Q: {query}
A:
""".strip()

In [5]:
from math import nan


def run_syncode(textual_query: str) -> tuple[list[float], str, float]:
    input = PROMPT.format(query=textual_query)
    output: str = llm.infer(input)[0]  # type: ignore

    left, op, right = re.split(r" ([+\-*/]) ", output)
    left, right = float(left), float(right)

    result = nan
    if op == "*":
        result = left * right
    elif op == "+":
        result = left + right
    elif op == "-":
        result = left - right
    elif op == "/":
        result = left / right

    return ([left, right], op, result)